# NPTEL Study Notes: Transformer Interpretability & Alternative Formulations

## 1. The Residual Stream Perspective

Traditional explanations of the Transformer treat layers as sequential pipelines where the input is continuously transformed and rewritten. The **Residual Stream Perspective** provides an alternative view critical for mechanistic interpretability: a Transformer is a single data pipeline (the residual stream) to which attention heads and Feed-Forward Networks (FFNs) independently **read from** and **write to** via additive operations.

* **Initial State:** The input token $x_i$ is mapped to a $d$-dimensional representation via the Embedding matrix ($W_E$).
* **The Stream:** The input forms a matrix $X \in \mathbb{R}^{n \times d}$. At each layer $l$, intermediate computations are strictly *added* to $X$.
* **Operations:** Both Multi-Head Self-Attention and FFNs are treated as separate additive components contributing to the overall vector of the token.
* **Final Output:** At the end of the network, the final residual stream state for a token is multiplied by the **Unembedding layer ($W_U$)** to convert the $d$-dimensional vector back into a vocabulary distribution to predict the next word.

## 2. Masked Self-Attention Review

In a decoder-only model (like GPT), a token must only attend to itself and previous tokens to prevent data leakage (predicting the future).

* **Mask Matrix ($M$):** An attention score matrix is computed for all tokens. The upper diagonal elements (representing future tokens, where $j > i$) are set to $-\infty$.
* **Softmax:** When passed through the softmax function, $e^{-\infty} = 0$, ensuring future tokens have an attention weight of exactly $0$.

### Dimension Matrix

| Component | Notation | Dimension | Description |
| --- | --- | --- | --- |
| **Input Sequence** | $X$ | $n \times d$ | $n$ tokens, each of dimension $d$. |
| **Token at layer $l-1$** | $x_i^{l-1}$ | $1 \times d$ | The $i$-th token representation before layer $l$. |
| **Query, Key, Value Projections** | $W_Q, W_K, W_V$ | $d \times d_h$ | Projects stream to head dimension ($d_h = d/H$). |
| **Output Projection** | $W_O$ | $d_h \times d$ | Projects head output back to stream dimension $d$. |

---

## 3. Decomposing Attention: QK and OV Circuits

To understand attention mechanistically, we decompose standard attention into two distinct, independently operating circuits per head.

### The Query-Key (QK) Circuit

The QK circuit determines **where** information should move (i.e., it computes the attention scores).

* **Formula:** $W_{QK} = W_Q W_K^T$
* **Mechanism:** The attention score between token $i$ and token $j$ is derived by reading token $i$'s query and token $j$'s key. Mathematically: $x_i^{l-1} W_Q W_K^T (x_j^{l-1})^T$.
* **Function:** This circuit acts as a router, generating the attention distribution grid.

### The Output-Value (OV) Circuit

The OV circuit determines **what** information is extracted and written back to the residual stream, independent of where it moves.

* **Formula:** $W_{OV} = W_V W_O$
* **Dimension:** $W_V \in \mathbb{R}^{d \times d_h}$ multiplied by $W_O \in \mathbb{R}^{d_h \times d}$ yields $W_{OV} \in \mathbb{R}^{d \times d}$.
* **Mechanism:** Once the QK circuit decides token $i$ should attend to token $j$ with probability $a_{i,j}$, the OV circuit takes token $j$'s representation, applies $W_{OV}$, and adds the result to token $i$'s residual stream.
* **Vector Output:** The output of head $h$ at layer $l$ is $\sum_{j \le i} a_{i,j} x_j^{l-1} W_{OV}^{l,h}$.

> **Note on Multi-Head Addition:** Because of the $W_O$ projection, each head produces a vector of size $d$. Instead of concatenating the outputs (which is mathematically equivalent but conceptually different), the residual stream perspective simply **sums** the outputs of all $H$ heads and adds them to the stream: $x_i^{mid} = x_i^{l-1} + \sum_{h=1}^H \text{Head}_{l,h}(x_{\le i}^{l-1})$.

---

## 4. The Feed-Forward Network (FFN) as Key-Value Memory

Following the attention block, the intermediate stream $x_i^{mid}$ is passed through a Multi-Layer Perceptron (MLP/FFN). Mechanistically, this layer acts as a continuous **Key-Value Memory**.

* **Upward Projection ($W_{in}$):** Size $d \times d_f$.
* **Function as "Key":** The columns of $W_{in}$ act as pattern detectors (Keys). When the intermediate vector $x_i^{mid}$ is multiplied by $W_{in}$, it computes how strongly the current stream matches specific predetermined features.


* **Non-linearity ($g$):** Generates Neuronal Activations.
* **Formula:** $N_u^l = g(x_i^{mid} W_{in}[:, u])$.
* $N_u^l$ is a scalar representing the activation of the $u$-th neuron, yielding a vector of size $d_f$.


* **Downward Projection ($W_{out}$):** Size $d_f \times d$.
* **Function as "Value":** $W_{out}$ stores vectors (Values) to be written back to the stream. The neuronal activations $N_u^l$ up-weight or down-weight the corresponding rows of $W_{out}$.



The final FFN operation is an element-wise mapping where active neurons trigger specific $d$-dimensional value vectors to be added to the residual stream: $x_i^l = x_i^{mid} + \sum_{u=1}^{d_f} N_u^l W_{out}[u, :]$.

---

## 5. Mathematical Decomposition of Multi-Layer Circuits

To see how a model builds complex behaviors, consider a simplified **2-layer transformer without FFNs** focusing on a single token $x$.

* **Layer 1 Output:** $x_1 = x + x W_{OV1}$
* **Layer 2 Output:** $x_2 = x_1 + x_1 W_{OV2}$
* **Substitution:** $x_2 = (x + x W_{OV1}) + (x + x W_{OV1}) W_{OV2}$
* **Expansion:** $x_2 = x + x W_{OV1} + x W_{OV2} + x W_{OV1} W_{OV2}$

When generating the final prediction, this $x_2$ vector is multiplied by the Unembedding layer ($W_U$). This expands into four distinct mechanistic paths:

1. **Direct Path:** $x W_U$ (The input embedding directly affects the output).
2. **Full OV Circuit 1:** $x W_{OV1} W_U$ (Information passes through layer 1's attention only).
3. **Full OV Circuit 2:** $x W_{OV2} W_U$ (Information passes through layer 2's attention only).
4. **Virtual Attention Head (V-Composition):** $x W_{OV1} W_{OV2} W_U$ (Information is processed by layer 1, written to the stream, and then that exact modification is read by layer 2).

> **Virtual Attention Head:** The matrix multiplication $W_{OV1} W_{OV2}$ acts mathematically exactly like a single, highly complex attention head. This shows how deeper layers compose lower-layer outputs into increasingly sophisticated features.

---

## 6. NPTEL Exam Practice Questions

**Q1: In the residual stream perspective, what is the dimension of the Output-Value (OV) circuit matrix $W_{OV}$, assuming the residual stream dimension is $d$ and head dimension is $d_h$?**
A) $d_h \times d_h$
B) $d \times d_h$
C) $d \times d$
D) $d_h \times d$
**Answer:** **C**
**Explanation:** The OV circuit is computed as $W_V W_O$. $W_V$ has dimensions $d \times d_h$ and $W_O$ has dimensions $d_h \times d$. Matrix multiplication ($d \times d_h \times d_h \times d$) results in a matrix of size $d \times d$.

**Q2: Which component of a Transformer layer acts as a "Key" matrix in a Key-Value memory setup by identifying specific patterns in the residual stream?**
A) The Key matrix ($W_K$) in self-attention
B) The Output matrix ($W_O$)
C) The upward projection matrix ($W_{in}$) in the FFN
D) The downward projection matrix ($W_{out}$) in the FFN
**Answer:** **C**
**Explanation:** While $W_K$ is used for routing attention, in the Key-Value memory interpretation of the Feed-Forward Network, the upward projection matrix $W_{in}$ acts as the "Key". Its columns detect specific patterns in the input vector $x_i^{mid}$, which then trigger neuronal activations.

**Q3: The mathematical term $x W_{OV1} W_{OV2} W_U$ in a simplified two-layer transformer represents which mechanistic concept?**
A) The Direct Path
B) QK Circuit composition
C) A Virtual Attention Head
D) Masked self-attention
**Answer:** **C**
**Explanation:** When the output of layer 1's OV circuit ($W_{OV1}$) is processed by layer 2's OV circuit ($W_{OV2}$), the combined sequential operation ($W_{OV1} W_{OV2}$) functions identically to a single, more complex attention head, termed a "Virtual Attention Head" or V-Composition.

**Q4: Why is upper-diagonal masking used in the Query-Key dot product of decoder-only transformers?**
A) To reduce computational complexity from $O(N^2)$ to $O(N)$
B) To force tokens to only attend to future context
C) To set the attention scores of future tokens to $-\infty$ so they become $0$ after softmax
D) To project the residual stream dimension $d$ to head dimension $d_h$
**Answer:** **C**
**Explanation:** Decoder-only models are autoregressive and cannot look ahead. Adding $-\infty$ to the upper diagonal of the attention score matrix ensures that when the softmax function is applied, the probability mass assigned to those future tokens is exactly 0.

---

## 7. Revision Summary

* **Residual Stream:** A $d$-dimensional bus where $X$ is constantly updated via addition ($X \leftarrow X + \text{Attention} + \text{FFN}$).
* **QK Circuit ($W_{QK} = W_Q W_K^T$):** Computes attention scores; decides **where** to move information.
* **OV Circuit ($W_{OV} = W_V W_O$):** Projects representations to $d \times d$; decides **what** information to write to the stream.
* **Multi-head Addition:** Instead of concatenation, individual head outputs (size $d$) are simply summed together to update the residual stream.
* **FFN as KV Memory:** $W_{in}$ extracts patterns (Keys) producing scalar activations, which up-weight rows of $W_{out}$ (Values) added back to the stream.
* **Virtual Attention:** Composition of sequential OV matrices ($W_{OV1} W_{OV2}$) allows later layers to perform complex higher-order reasoning over outputs generated by previous layers.